In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#!ls "/content/drive/MyDrive/Colab Notebooks/"

In [3]:
# !pip install tensorflow==2.19.0
# import tensorflow as tf
# print(tf.__version__)

2.19.0


In [7]:
import pandas as pd
import joblib
import numpy as np
from tensorflow.keras.models import load_model

FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/"

# ===============================
# LOAD DLQ STREAM
# ===============================
df = pd.read_csv(FILE_PATH + "windows_dlq_stream_test.csv")

print("DLQ events:", len(df))

# ===============================
# FEATURE ENGINEERING
# ===============================
df["date_and_time"] = pd.to_datetime(df["date_and_time"])

df = df.sort_values("date_and_time")

df["hour"] = df["date_and_time"].dt.hour
df["night_activity"] = ((df["hour"] < 5) | (df["hour"] > 23)).astype(int)
df["is_night"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)
# Consistent with training
df["event_freq"] = df.groupby("event_id").cumcount()
df["burst_count"] = df["event_freq"].clip(upper=10)
#df["time_diff"] = df["date_and_time"].diff().dt.total_seconds().fillna(0)

# IMPORTANT: removed event_id
features = [
    "event_id",
    "event_freq",
    "burst_count",
    "hour",
    "is_night"
]

X = df[features]

# ===============================
# LOAD MODEL
# ===============================
model = load_model(FILE_PATH + "windows_mlp_model.h5")
scaler = joblib.load(FILE_PATH + "windows_mlp_scaler.pkl")

# Ensure column order
if hasattr(scaler, "feature_names_in_"):
    X = X.reindex(columns=scaler.feature_names_in_, fill_value=0)
X = scaler.transform(X)

# ===============================
# PREDICT
# ===============================
probs = model.predict(X)

# Dynamic threshold
threshold = np.percentile(probs, 95)

df["mlp_anomaly"] = (probs.flatten() > threshold).astype(int)

# ===============================
# OUTPUT
# ===============================
anomalies = df[df["mlp_anomaly"] == 1]

print("\nMLP anomalies detected:", len(anomalies))
print(anomalies.head())

# ===============================
# SAVE
# ===============================
anomalies.to_csv(
    FILE_PATH + "windows_mlp_alerts.csv",
    index=False
)

print("MLP analysis completed")

DLQ events: 100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

MLP anomalies detected: 5
         date_and_time                               source  event_id  \
5  2024-10-29 17:10:12  Microsoft-Windows-Security-Auditing      4624   
44 2024-11-13 17:32:59  Microsoft-Windows-Security-Auditing      4799   
45 2024-11-13 17:38:33  Microsoft-Windows-Security-Auditing      4799   
48 2024-11-14 20:48:39  Microsoft-Windows-Security-Auditing      4799   
49 2024-11-14 20:55:20  Microsoft-Windows-Security-Auditing      4799   

                task_category  \
5                       Logon   
44  Security Group Management   
45  Security Group Management   
48  Security Group Management   
49  Security Group Management   

                                              message  hour  night_activity  \
5   An account was successfully logged on.\r\n\r\n...    17               0   
44  A security-enabled local group membership was ...    17               0   
45  A security-enabled local group membershi

In [4]:
# import pandas as pd
# import joblib
# import numpy as np
# from tensorflow.keras.models import load_model

# FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# # ===============================
# # LOAD DLQ STREAM
# # ===============================

# df=pd.read_csv(FILE_PATH+"windows_dlq_stream_test.csv")

# print("DLQ events:",len(df))

# # ===============================
# # FEATURE ENGINEERING
# # ===============================

# df["date_and_time"]=pd.to_datetime(df["date_and_time"])

# df["hour"]=df["date_and_time"].dt.hour

# df["is_night"]=((df["hour"]<6)|(df["hour"]>22)).astype(int)

# features=[

# "event_id",
# "event_freq",
# "burst_count",
# "hour",
# "is_night"

# ]

# X=df[features]

# # ===============================
# # LOAD MODEL
# # ===============================

# model=load_model(FILE_PATH+"windows_mlp_model.h5")

# scaler=joblib.load(FILE_PATH+"windows_mlp_scaler.pkl")

# X=scaler.transform(X)

# # ===============================
# # PREDICT
# # ===============================

# probs=model.predict(X)

# df["mlp_anomaly"]=0
# threshold = np.percentile(probs, 95)  # top 5% anomalies
# df["mlp_anomaly"] = (probs.flatten() > threshold).astype(int)
# # ===============================
# # EXTRACT MLP ANOMALIES
# # ===============================

# anomalies = df[df["mlp_anomaly"] == 1]

# print("\nMLP anomalies detected:", len(anomalies))
# print(anomalies.head())

# # ===============================
# # SAVE ANOMALY FILE
# # ===============================

# anomalies.to_csv(
# FILE_PATH+"windows_mlp_alerts.csv",
# index=False
# )

# # print("MLP alerts saved:",len(anomalies))
# # # ===============================
# # # PRINT ANOMALIES
# # # ===============================

# # anomalies=df[df["mlp_anomaly"]==1]

# # print("\nMLP anomalies detected:",len(anomalies))

# # print(anomalies[[
# # "date_and_time",
# # "event_id",
# # "risk_score",
# # "mlp_anomaly"
# # ]].head())

# # # ===============================
# # # SAVE
# # # ===============================

# #df.to_csv(FILE_PATH+"windows_mlp_output.csv",index=False)

# print("MLP analysis completed")

DLQ events: 144


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

MLP anomalies detected: 126
         date_and_time                               source  event_id  \
6  2024-10-29 15:29:01  Microsoft-Windows-Security-Auditing      4624   
7  2024-10-29 16:48:01  Microsoft-Windows-Security-Auditing      4624   
8  2024-10-29 17:10:12  Microsoft-Windows-Security-Auditing      4624   
13 2024-11-01 02:59:36  Microsoft-Windows-Security-Auditing      4672   
14 2024-11-01 23:04:40  Microsoft-Windows-Security-Auditing      4672   

    task_category                                            message  hour  \
6           Logon  An account was successfully logged on.\r\n\r\n...    15   
7           Logon  An account was successfully logged on.\r\n\r\n...    16   
8           Logon  An account was successfully logged on.\r\n\r\n...    17   
13  Special Logon  Special privileges assigned to new logon.\r\n\...     2   
14  Special Logon  Special privileges assigned to new logon.\r\n\...    23   

    night_activity  event